# Sparrow IPC: Arrow files between C++ and pyarrow

This notebook mixes C++ and Python thanks to the [polyglott kernel](https://github.com/DerThorsten/jupyterlite-polyglott):
the first line of each cell, `%%kernel xcpp20` or `%%kernel xpython`, selects the language.

C++ and Python run in two separate WebAssembly kernels that share the browser file system:
[sparrow-ipc](https://github.com/sparrow-org/sparrow-ipc) writes an Arrow IPC stream file that pyarrow reads (`from_cpp.arrows`),
and pyarrow writes one that sparrow-ipc reads (`from_python.arrows`).

The demo notebooks: [sparrow.ipynb](sparrow.ipynb) (arrays), [sparrow-ipc.ipynb](sparrow-ipc.ipynb) (Arrow IPC with pyarrow), [sparrow-compute.ipynb](sparrow-compute.ipynb) (lazy compute) and [rockfinch.ipynb](rockfinch.ipynb) (C++ extension module for Python).

Run the cells from top to bottom (`Shift+Enter`). A C++ name can only be defined once per kernel: reload the page before re-running the notebook.

In [ ]:
%%kernel xcpp20
#pragma clang diagnostic ignored "-W#warnings"
// Must match the compile definitions of the WebAssembly sparrow package
#define SPARROW_USE_LARGE_INT_PLACEHOLDERS
#include <dlfcn.h>
#include <format>
#include <fstream>
#include <iostream>
#include <iterator>
#include <string>
#include <vector>

// <sparrow.hpp> loads libsparrow.so by itself; sparrow-ipc must be loaded explicitly
if (dlopen("/lib/libsparrow-ipc.so", RTLD_NOW | RTLD_GLOBAL) == nullptr)
{
    std::cout << dlerror() << std::endl;
}

#include <sparrow.hpp>
#include <sparrow/record_batch.hpp>
#include <sparrow/config/sparrow_version.hpp>
#include <sparrow_ipc/deserialize.hpp>
#include <sparrow_ipc/memory_output_stream.hpp>
#include <sparrow_ipc/serializer.hpp>
#include <sparrow_ipc/config/sparrow_ipc_version.hpp>

namespace sp = sparrow;
namespace sp_ipc = sparrow_ipc;

std::cout << "sparrow " << sp::SPARROW_VERSION_MAJOR << "." << sp::SPARROW_VERSION_MINOR << "." << sp::SPARROW_VERSION_PATCH << std::endl;
std::cout << "sparrow-ipc " << sp_ipc::SPARROW_IPC_VERSION_MAJOR << "." << sp_ipc::SPARROW_IPC_VERSION_MINOR << "." << sp_ipc::SPARROW_IPC_VERSION_PATCH << std::endl;

In [ ]:
%%kernel xpython
import pyarrow as pa
import polars as pl

print("pyarrow", pa.__version__, "| polars", pl.__version__)

## From C++ to Python

sparrow builds a record batch, sparrow-ipc serializes it to an Arrow IPC stream.

In [ ]:
%%kernel xcpp20
sp::record_batch rb(
    {{"id", sp::array(sp::primitive_array<int32_t>{1, 2, 3, 4})},
     {"value", sp::array(sp::primitive_array<double>{1.5, 2.5, 3.5, 4.5})},
     {"name", sp::array(sp::string_array(std::vector<std::string>{"alpha", "beta", "gamma", "delta"}))}});
std::vector<sp::record_batch> batches;
batches.push_back(std::move(rb));

std::vector<uint8_t> cpp_stream;
sp_ipc::memory_output_stream stream(cpp_stream);
sp_ipc::serializer serializer(stream);
serializer << batches << sp_ipc::end_stream;

std::ofstream cpp_file("from_cpp.arrows", std::ios::binary);
cpp_file.write(reinterpret_cast<const char*>(cpp_stream.data()), cpp_stream.size());
cpp_file.close();
std::cout << "Wrote " << cpp_stream.size() << " bytes to from_cpp.arrows" << std::endl;

In [ ]:
%%kernel xpython
# pyarrow memory-maps file paths, which the browser file system does not support: read the bytes instead
with open("from_cpp.arrows", "rb") as f:
    table = pa.ipc.open_stream(f.read()).read_all()
print(table)

In [ ]:
%%kernel xpython
df = pl.from_arrow(table)
print(df)

## From Python to C++

Polars transforms the data, pyarrow writes it back as an Arrow IPC stream, sparrow-ipc reads it in C++.

In [ ]:
%%kernel xpython
result = df.filter(pl.col("id") % 2 == 0).with_columns((pl.col("value") * 10).alias("value_x10"))
print(result)

# pa.table() imports the DataFrame through the Arrow PyCapsule Interface (__arrow_c_stream__);
# result.to_arrow() panics in the WebAssembly build of Polars
result_table = pa.table(result)
with pa.OSFile("from_python.arrows", "wb") as f:
    with pa.ipc.new_stream(f, result_table.schema) as writer:
        writer.write_table(result_table)
print(result_table.schema)

In [ ]:
%%kernel xcpp20
void print_batch(const sp::record_batch& batch)
{
    for (std::size_t i = 0; i < batch.nb_columns(); ++i)
    {
        std::cout << std::format("{}", batch.get_column(i)) << std::endl;
    }
}

In [ ]:
%%kernel xcpp20
std::ifstream python_file("from_python.arrows", std::ios::binary);
std::vector<uint8_t> python_stream((std::istreambuf_iterator<char>(python_file)), std::istreambuf_iterator<char>());
std::vector<sp::record_batch> from_python = sp_ipc::deserialize_stream(std::span<const uint8_t>(python_stream));
std::cout << "Read " << from_python.size() << " batch, " << from_python[0].nb_rows() << " rows" << std::endl;
print_batch(from_python[0]);